### CELL 1: Import thư viện

In [ ]:
# --- CẤU HÌNH & IMPORT THƯ VIỆN ---
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Import từ các file utils của project hiện tại
from config import *
from data_utils import *
from model_utils import *
from dl_model import build_bilstm_model, build_cnn_model

print("Đã load xong thư viện và cấu hình hệ thống!")

### CELL 2: Đọc và Tiền xử lý dữ liệu (ETL & Text Cleaning)

In [ ]:
print("Đang kiểm tra dữ liệu Deep Learning đã tiền xử lý...")

if os.path.exists(DL_PROCESSED_DATA_PATH):
    df = pd.read_csv(DL_PROCESSED_DATA_PATH)
    print("Đã load dữ liệu dl_processed thành công từ bộ nhớ đệm!")
else:
    print("Không tìm thấy dl_processed. Bắt đầu xử lý từ dữ liệu thô (Raw Data)...")
    df_fake = pd.read_csv(RAW_FAKE_PATH)
    df_true = pd.read_csv(RAW_TRUE_PATH)
    df_fake['label'] = 1
    df_true['label'] = 0
    df = pd.concat([df_fake, df_true], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    
    # Loại bỏ giá trị Null ở cả 2 cột để tránh lỗi ép kiểu chuỗi
    df = df.dropna(subset=['title', 'text']).reset_index(drop=True)
    
    print("Đang thực hiện gộp Title + Text và chạy quy trình deep_clean_text...")
    # GỘP TIÊU ĐỀ VÀ NỘI DUNG
    df['title_text'] = df['title'].astype(str) + " " + df['text'].astype(str)
    df['text_clean_raw'] = df['title_text'].apply(deep_clean_text)
    
    os.makedirs(os.path.dirname(DL_PROCESSED_DATA_PATH), exist_ok=True)
    df.to_csv(DL_PROCESSED_DATA_PATH, index=False)
    print(f"Quy trình hoàn tất! Đã tạo và lưu file tại: {DL_PROCESSED_DATA_PATH}")

# Đoạn kiểm tra phòng ngừa lỗi bộ đệm dữ liệu cũ
if 'title_text' not in df.columns or 'text_clean_raw' not in df.columns:
    print("Phát hiện file cache cũ không tương thích. Đang xử lý gộp Title + Text...")
    df = df.dropna(subset=['title', 'text']).reset_index(drop=True)
    df['title_text'] = df['title'].astype(str) + " " + df['text'].astype(str)
    df['text_clean_raw'] = df['title_text'].apply(deep_clean_text)

# Làm sạch khoảng trắng phát sinh
df['text_clean_raw'] = df['text_clean_raw'].fillna('').astype(str).str.strip()
df = df[df['text_clean_raw'] != ''].reset_index(drop=True)

# Xóa trùng lặp sâu dựa trên văn bản đã gộp sạch để triệt tiêu Data Leakage
df = df.drop_duplicates(subset=['text_clean_raw']).reset_index(drop=True)

# Tạo cột text đã lemmatize và loại stopwords phục vụ mạng Học sâu
print("Đang chuẩn hóa sâu bằng lemmatization và stopword removal...")
df['text_lemmatized'] = df['text_clean_raw'].apply(lemmatize_and_remove_stopwords)

if 'text_only_clean' not in df.columns:
    df['text_only_clean'] = df['text_lemmatized']

# Chia tập dữ liệu tường minh: Train (70%), Validation (15%), Test (15%)
print("Đang phân tách dữ liệu (Train/Val/Test = 70/15/15)...")
train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=RANDOM_STATE, stratify=df['label'])
train_df, val_df = train_test_split(train_val_df, test_size=0.1765, random_state=RANDOM_STATE, stratify=train_val_df['label'])

X_train_text, y_train = train_df['text_lemmatized'].astype(str).values, train_df['label'].values
X_val_text, y_val = val_df['text_lemmatized'].astype(str).values, val_df['label'].values
X_test_text, y_test = test_df['text_lemmatized'].astype(str).values, test_df['label'].values

print("-" * 40)
print(f"Kích thước tập Train: {len(X_train_text):,} mẫu")
print(f"Kích thước tập Validation: {len(X_val_text):,} mẫu")
print(f"Kích thước tập Test: {len(X_test_text):,} mẫu")

print("\nĐang xuất biểu đồ Phân tích dữ liệu Khám phá (EDA)...")
plot_dl_eda(df)

### CELL 3: Tokenization & Padding (Chuẩn bị đầu vào NLP)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

print("Đang thực hiện Tokenization & Padding cho cả 3 tập...")
# 1. Khởi tạo và Fit tokenizer trên tập Train
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

# Lưu Tokenizer
with open(TOKENIZER_PATH, 'wb') as f:
    pickle.dump(tokenizer, f)

# 2. Chuyển đổi và Padding cho cả 3 tập (Train, Val, Test)
X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(X_val_text), maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

vocab_size = min(len(tokenizer.word_index) + 1, MAX_WORDS)
print(f"Kích thước bộ từ vựng (Vocabulary Size): {vocab_size}")

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
# Dừng sớm nếu sau 4 epoch không tăng
early_stopping = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
# Giảm tốc độ học đi một nửa nếu chững lại
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=0.00001, verbose=1)

### CELL 4: Huấn luyện Kiến trúc Bi-LSTM (Mạng bộ nhớ ngắn-dài 2 chiều)

In [ ]:
print("KHỞI TẠO KIẾN TRÚC BI-LSTM...")
bilstm_model = build_bilstm_model(vocab_size, EMBEDDING_DIM, MAX_SEQUENCE_LENGTH)
bilstm_model.summary()

checkpoint_bilstm = ModelCheckpoint(BILSTM_MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)

print("\nBẮT ĐẦU HUẤN LUYỆN BI-LSTM...")
history_bilstm = bilstm_model.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_pad, y_val), # Truyền tập Validation độc lập
    callbacks=[early_stopping, lr_scheduler, checkpoint_bilstm], # Bổ sung lr_scheduler
    verbose=1
)

### CELL 5: Huấn luyện Kiến trúc 1D CNN

In [ ]:
print("KHỞI TẠO KIẾN TRÚC 1D CNN...")
cnn_model = build_cnn_model(vocab_size, EMBEDDING_DIM, MAX_SEQUENCE_LENGTH)
cnn_model.summary()

checkpoint_cnn = ModelCheckpoint(CNN_MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)

print("\nBẮT ĐẦU HUẤN LUYỆN 1D CNN...")
history_cnn = cnn_model.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_pad, y_val),
    callbacks=[early_stopping, lr_scheduler, checkpoint_cnn],
    verbose=1
)

### CELL 6: Trực quan hóa và Đánh giá

In [ ]:
print("Đang tính toán xác suất dự đoán trên tập kiểm thử (Test Set)...")
y_pred_probs_bilstm = bilstm_model.predict(X_test_pad)
y_pred_probs_cnn = cnn_model.predict(X_test_pad)

y_pred_class_bilstm = (y_pred_probs_bilstm >= 0.5).astype(int).flatten()
y_pred_class_cnn = (y_pred_probs_cnn >= 0.5).astype(int).flatten()

print("\n" + "="*60)
print("CLASSIFICATION REPORT: MÔ HÌNH BI-LSTM")
print("="*60)
print(classification_report(y_test, y_pred_class_bilstm, target_names=['Tin thật (0)', 'Tin giả (1)']))

print("\n" + "="*60)
print("CLASSIFICATION REPORT: MÔ HÌNH 1D CNN")
print("="*60)
print(classification_report(y_test, y_pred_class_cnn, target_names=['Tin thật (0)', 'Tin giả (1)']))

# Vẽ biểu đồ cũ (Loss & ROC-AUC)
print("\nĐang xuất biểu đồ Lịch sử huấn luyện (Loss & Accuracy) của Bi-LSTM...")
plot_dl_history(history_bilstm)

print("\nĐang xuất biểu đồ so sánh đường cong tương quan ROC-AUC...")
models_comparison_dict = {
    "Bi-LSTM Architecture": y_pred_probs_bilstm,
    "1D CNN Architecture": y_pred_probs_cnn
}
plot_multiple_roc_auc(models_comparison_dict, y_test)

# Vẽ biểu đồ mới (PR Curve & Confidence Histogram)
print("\nĐang xuất biểu đồ Precision-Recall Curve (Mô hình Bi-LSTM)...")
plot_pr_curve(y_test, y_pred_probs_bilstm, "Bi-LSTM")

print("\nĐang xuất biểu đồ Phân phối Độ tự tin (Confidence Histogram)...")
plot_prediction_confidence(y_test, y_pred_probs_bilstm, "Bi-LSTM")

### CELL 7: Confusion Matrix

In [ ]:
print("\n" + "="*60)
print("TRỰC QUAN HÓA MA TRẬN NHẦM LẪN (CONFUSION MATRIX)")
print("="*60)

# Vẽ Confusion Matrix cho Bi-LSTM
print("\nĐang vẽ Confusion Matrix cho Bi-LSTM...")
plot_dl_confusion_matrix(y_test, y_pred_class_bilstm, model_name="Bi-LSTM")

# Vẽ Confusion Matrix cho 1D CNN
print("\nĐang vẽ Confusion Matrix cho 1D CNN...")
plot_dl_confusion_matrix(y_test, y_pred_class_cnn, model_name="1D CNN")